# GlowLab · 02 — MobileNetV2 transfer learning

"Kod satrlariga izohlar"dagi food detector bilan bir xil g'oya: tayyor **MobileNetV2** (ImageNet'da o'qitilgan) → **backbone muzlatiladi** → yangi `classifier` faqat bizning 6 sinf uchun o'qitiladi → oxirida oxirgi bloklar past `lr` bilan **fine-tune** qilinadi.

Runtime → Change runtime type → **GPU** ni tanlang.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/glowlab_ml'
DATA = f'{ROOT}/data'
OUT = f'{ROOT}/models'          # O'qitilgan model shu yerga saqlanadi
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
import json, time, copy
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models

device = 'cuda' if torch.cuda.is_available() else 'cpu'   # GPU bo'lsa GPU'da
print('device:', device)
CLASSES = ['cleanser', 'toner', 'serum', 'moisturizer', 'sunscreen', 'lip']  # Ilovadagi kategoriyalar bilan bir xil nomlar (tartib MUHIM!)

In [ ]:
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet normalizatsiyasi (ilovada ham AYNAN shu)

# Augmentation faqat train uchun — model har xil burchak/yorug'likka o'rganadi
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
# Val/test: augmentation yo'q, faqat o'lcham + normalizatsiya (ilovadagi preprocessing bilan bir xil)
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(f'{DATA}/train', transform=train_tf)   # Papka nomi = sinf
val_ds = datasets.ImageFolder(f'{DATA}/val', transform=eval_tf)

# ImageFolder sinflarni alifbo tartibida beradi → bizning CLASSES tartibiga keltiramiz
assert sorted(CLASSES) == train_ds.classes, train_ds.classes
remap = {train_ds.class_to_idx[c]: i for i, c in enumerate(CLASSES)}
train_ds.target_transform = lambda t: remap[t]
val_ds.target_transform = lambda t: remap[t]

# Sinflar nomutanosib bo'lishi mumkin (masalan lip ko'p, serum kam) → kam sinfga kattaroq og'irlik
from collections import Counter
cnt = Counter(remap[t] for t in train_ds.targets)
class_weights = torch.tensor([len(train_ds) / (len(CLASSES) * cnt[i]) for i in range(len(CLASSES))],
                             dtype=torch.float32).to(device)
print('train per class:', {c: cnt[i] for i, c in enumerate(CLASSES)})

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
print(len(train_ds), 'train,', len(val_ds), 'val')

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)   # Tayyor og'irliklar

for p in model.features.parameters():
    p.requires_grad = False          # Backbone muzlatildi — faqat yangi qatlam o'rganadi

model.classifier[1] = nn.Linear(model.last_channel, len(CLASSES))   # 1280 → 6 sinf
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('trainable params:', trainable)

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    """Bitta epoch: optimizer berilsa — o'qitish, aks holda — baholash"""
    training = optimizer is not None
    model.train() if training else model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(training):              # Baholashda gradient hisoblanmaydi (tezroq)
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
    return loss_sum / total, correct / total

def fit(model, epochs, lr):
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)   # Og'irlikli loss
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    best_acc, best_state = 0.0, None
    for epoch in range(epochs):
        t = time.time()
        tr_loss, tr_acc = run_epoch(model, train_dl, loss_fn, optimizer)
        va_loss, va_acc = run_epoch(model, val_dl, loss_fn)
        if va_acc > best_acc:                                  # Eng yaxshi val natijani eslab qolamiz
            best_acc, best_state = va_acc, copy.deepcopy(model.state_dict())
        print(f'{epoch+1:02d} train {tr_acc:.3f} | val {va_acc:.3f} (loss {va_loss:.3f}) | {time.time()-t:.0f}s')
    model.load_state_dict(best_state)                          # Oxirgisini emas, eng yaxshisini qaytaramiz
    return best_acc

In [ ]:
# 1-bosqich: faqat classifier (tez)
best1 = fit(model, epochs=8, lr=1e-3)
print('best val (head only):', round(best1, 3))

In [ ]:
# 2-bosqich: fine-tuning — oxirgi 4 blokni ochamiz, juda kichik lr bilan (eski bilim buzilmasin)
for p in model.features[-4:].parameters():
    p.requires_grad = True
best2 = fit(model, epochs=6, lr=1e-4)
print('best val (fine-tuned):', round(best2, 3))

In [ ]:
torch.save(model.state_dict(), f'{OUT}/product_classifier.pt')   # Faqat og'irliklar (state_dict)
with open(f'{OUT}/classes.json', 'w') as f:
    json.dump(CLASSES, f)                                           # Sinflar tartibi — ilova ham shuni o'qiydi
print('saved to', OUT)